<a href="https://colab.research.google.com/github/lim0119/-2025-3-2-PJ/blob/main/%EA%B8%B0%EB%A7%90_%ED%92%88%EC%A7%88_%EC%BD%94%EB%93%9C(%ED%86%B5%ED%95%A9_%ED%85%8C%EC%8A%A4%ED%8A%B8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile test_quality_integration_runner.py
import pytest
import numpy as np
import random
import os
from qa_metrics import main_smoke_test
from typing import List


# -------------------- 더미 데이터 생성 (실제 통합 테스트 데이터 대체) --------------------
def create_integration_data():
    """최종 테스트 데이터 모방."""

    # 통합 테스트에서 '전체 파이프라인이 돌 수 있는 입력 구조'를 만들기 위한 더미 데이터 생성.
    # 모델 입력, 환자 ID, 라벨, 메타데이터가 모두 포함된 실제 서비스 환경을 흉내냄.

    N = 100
    pids = [f"P_{i:04d}" for i in range(N)]
    labels = [1 if i % 3 == 0 else 0 for i in range(N)]
    images = [np.random.rand(64, 64, 64).astype(np.float32) for _ in pids]

    # 메타데이터는 공정성 분석에서 그룹별 지표를 비교하기 위해 필요.
    meta = []
    for i in range(N):
        gender = '남성' if i < N*0.6 else '여성'
        age = random.randint(60, 95)
        meta.append({"gender": gender, "age": age})

    items = list(zip(images, labels, pids, meta))

    # 실제 모델의 예측 함수 대신 확률을 랜덤으로 반환하는 더미 함수.
    # 통합 QA 파이프라인이 "모델 예측 함수 호출 → 지표 계산" 흐름을 정상적으로 수행하는지 확인하기 위함.
    def model_predict_fn(images: List[np.ndarray]) -> List[float]:
        return [random.uniform(0.1, 0.9) for _ in images]

    return items, model_predict_fn


def test_full_integration_quality_run():
    """전체 QA 파이프라인(main_smoke_test) 실행 및 최종 품질 지표 검증"""

    # 이 테스트는 '전체 시스템이 끝에서 끝까지(End-to-End) 정상 실행되는지' 확인하는 통합 테스트.
    # 단위 테스트에서는 잡히지 않는 파이프라인 전체의 오류를 조기에 찾아내는 목적.
    report = main_smoke_test(items, model_predict_fn)

    items, model_predict_fn = create_integration_data()


    # A. 데이터 누수 확인
    # 환자 ID가 train/test 사이에서 섞였는지, 인덱싱 오류가 있는지 등
    # '치명적인 QA 문제'가 없는지 통합적으로 확인.
    assert report['데이터 누수 확인'] is True, "통합 QA 실패: 데이터 누수 발생 (치명적 오류)"


    # B. 임상 목표 지표 확인
    # 특정 성능 조건(특이도 95%)에서 모델이 요구되는 민감도(>= 0.5)를 만족하는지 확인.
    # 임상적 안전성/성능 기준 충족 여부를 QA 단계에서 자동으로 검증하는 목적.
    spec_95_metrics = report['임상 목표 지표']['특이도_95%']
    assert spec_95_metrics['달성 민감도'] >= 0.5, \
        "통합 QA 실패: 임상 목표 특이도 95%에서 민감도 기준(0.5) 미달"


    # C. 공정성(Fairness) 검증
    # 성별 간 재현율 격차가 지나치게 크지 않은지 확인.
    # 모델이 특정 그룹에 불리한 bias를 가지고 있는지 자동 검출하기 위함.
    gender_metrics = report['그룹 성능 편향성']['성별_성능']

    if '남성' in gender_metrics and '여성' in gender_metrics:
        male_recall = gender_metrics['남성']['재현율']
        female_recall = gender_metrics['여성']['재현율']

        # 그룹 간 성능 차이가 기준(10%)을 넘지 않는지 확인.
        assert abs(male_recall - female_recall) <= 0.1, \
            "통합 QA 실패: 성별 간 재현율 격차(공정성) 기준 초과"
